# IMDB Movie Review Sentiment Analysis using Transformers

In this notebook we build a small transformer encoder from scratch and train it to decide whether a movie review is positive or negative. We use exactly the same data pipeline as notebook 12.5 (same tokenizer, vocabulary, train/validation split, and `MAXLEN`), so that we can compare the transformer with the LSTM at the end. We also run an ablation: what happens if the transformer is not told where each token is in the review?

## Step 1: Import Necessary Libraries and Setting Up the Environment

In this section, we import all the necessary libraries required for our sentiment classification task using Transformers in PyTorch. We start by importing `torch` and its submodules like `nn` for neural network layers, `optim` for optimization algorithms, and the `DataLoader` and `Dataset` classes from `torch.utils.data`, together with `pad_sequence` for padding sequences to a uniform length.

The data comes from the Hugging Face `datasets` library (`load_dataset`). We build the tokenizer and the vocabulary ourselves with Python's `re` (regular expressions) and `collections` modules, exactly as in notebook 12.5.

We also import `numpy`, a fundamental package for scientific computing, and `time` and `copy` to time the training and keep a copy of the best model.

Finally, we set the `device` variable to use a GPU if available (`cuda`), otherwise fallback to using the CPU. This ensures that our model training and inference processes are as efficient as possible, leveraging hardware acceleration when possible.

In [1]:
# Run this if you're in Colab (the datasets library is usually preinstalled there)
# %pip install datasets

In [2]:
import re
import collections
import copy
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2: Prepare the Data

This step is identical to notebook 12.5.

### Setting the Seed for Reproducibility
To make the experiment repeatable, we set a random seed for PyTorch and NumPy. This controls the randomness in weight initialization, dropout, data shuffling, and the train/validation split. We also ask CuDNN to use deterministic algorithms.

### Loading the IMDB Dataset
We load the IMDB movie review dataset from the Hugging Face hub with `load_dataset("imdb")`. It contains 25,000 training reviews and 25,000 test reviews, each labeled 0 (negative) or 1 (positive), with the two classes balanced (it also contains 50,000 unlabeled reviews that we do not use). The first call downloads the data (about 80 MB), so it needs internet access.

### Training, Validation, and Test Sets
We hold out 10% of the training split (2,500 reviews, chosen with a fixed seed) as a **validation set**. We use it to monitor the model after every epoch and to pick the best epoch. The **test set** is used only once, at the very end, to report the final accuracy. If we used the test set to monitor training and choose the epoch, the reported test accuracy would be optimistically biased.

### Tokenization and Vocabulary Building
A tokenizer splits each review into a list of tokens. We use a simple regular expression: after lowercasing the text and removing the HTML line breaks (`<br />`) that appear in many reviews, every run of letters, digits, and apostrophes becomes a word token and every other non-space character (such as punctuation) becomes a token of its own. For example, `"Don't watch it!"` becomes `["don't", "watch", "it", "!"]`.

We then build a **vocabulary** that maps each token to an integer index. The vocabulary is built from the training reviews only (never from validation or test data), and keeps the `MAX_VOCAB` = 20,000 most frequent tokens that occur at least `MIN_FREQ` = 5 times. Two special tokens are added: `<pad>` (index 0), used to fill short sequences, and `<unk>` (index 1), which replaces every token that is not in the vocabulary.

### Data Processing Functions
The `encode` function converts a raw review into a tensor of token indices and truncates it to at most `MAXLEN` tokens. We encode every review once, before training. The `collate_batch` function combines a list of encoded reviews into a batch: it pads them with `<pad>` to the length of the longest review in the batch and also returns the true (unpadded) length of each review, which the model needs in order to ignore the padding.

### DataLoader Setup
Finally, we wrap the encoded reviews in a small PyTorch `Dataset` class and create a `DataLoader` for each of the three sets. Only the training loader shuffles the data.

In [3]:
# Hyperparameters
BATCH_SIZE = 64
MAXLEN = 500      # Maximum length of a review (in tokens); longer reviews are truncated
MAX_VOCAB = 20000 # Keep at most this many of the most frequent tokens
MIN_FREQ = 5      # Ignore tokens that appear fewer than this many times in the training data

# Set seed for reproducibility
SEED = 1234
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# Load the IMDB dataset (labels: 0 = negative, 1 = positive)
ds = load_dataset("imdb")

# Hold out 10% of the training split as a validation set
split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_data, valid_data, test_data = split["train"], split["test"], ds["test"]
print(f"Training reviews: {len(train_data)}, validation reviews: {len(valid_data)}, test reviews: {len(test_data)}")

# Tokenizer: lowercase, remove HTML line breaks, split into words and punctuation
def tokenize(text):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9']", text.lower().replace("<br />", " "))

print(tokenize("Don't watch it!<br />It was 2 hours of my life..."))

# Build the vocabulary from the training reviews only
counter = collections.Counter()
for text in train_data["text"]:
    counter.update(tokenize(text))
PAD_IDX, UNK_IDX = 0, 1
itos = ["<pad>", "<unk>"] + [tok for tok, freq in counter.most_common(MAX_VOCAB) if freq >= MIN_FREQ]
stoi = {tok: idx for idx, tok in enumerate(itos)}
vocab_size = len(itos)
print(f"Distinct tokens in the training reviews: {len(counter)}, vocabulary size: {vocab_size}")


# Function to convert a review into a tensor of at most MAXLEN token indices
def encode(text):
    ids = [stoi.get(tok, UNK_IDX) for tok in tokenize(text)]
    return torch.tensor(ids[:MAXLEN], dtype=torch.int64)


# Encode every review once and keep the results in memory
class IMDBDataset(Dataset):
    def __init__(self, hf_data):
        self.texts = [encode(text) for text in hf_data["text"]]
        self.labels = hf_data["label"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.labels[idx], self.texts[idx]


# Function to collate data samples into batch tensors
def collate_batch(batch):
    labels = torch.tensor([label for label, _ in batch], dtype=torch.int64)
    text_list = [text for _, text in batch]
    lengths = torch.tensor([text.size(0) for text in text_list], dtype=torch.int64)
    texts = pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    return labels, texts, lengths


train_dataset = IMDBDataset(train_data)
valid_dataset = IMDBDataset(valid_data)
test_dataset = IMDBDataset(test_data)

all_lengths = np.array([len(tokenize(text)) for text in train_data["text"]])
print(f"Median review length: {np.median(all_lengths):.0f} tokens; {np.mean(all_lengths > MAXLEN)*100:.1f}% of the reviews are truncated to {MAXLEN} tokens")

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

Training reviews: 22500, validation reviews: 2500, test reviews: 25000
["don't", 'watch', 'it', '!', 'it', 'was', '2', 'hours', 'of', 'my', 'life', '.', '.', '.']


Distinct tokens in the training reviews: 83341, vocabulary size: 20002


Median review length: 201 tokens; 11.3% of the reviews are truncated to 500 tokens


## Step 3: Define the Transformer Model

In this section, we define the transformer model for sentiment classification.

<img src="images/scaled-dot-product-attention-recap.png" alt="scaled dot-product attention recap" width="800"/>

<img src="images/multi-head-attention.png" alt="multi head self attention" width="800"/>

### Scaled Dot-Product Attention

The `DotProductAttention` class implements the core attention mechanism that allows the model to focus on different parts of the input sequence when making predictions. This mechanism calculates attention scores by performing a dot product between query and key vectors, scaled by the square root of their dimension (the dimension of one head). The resulting scores are then used to create a weighted sum of the value vectors, determining the output of the attention layer.

The scaled dot-product attention operates on three vectors: queries (Q), keys (K), and values (V). The process involves the following steps:

1. **Linear Transformation**: Each word's embedding is transformed into three vectors (Q, K, V) using different weight matrices $W^Q, W^K, W^V$. This is represented by matrix multiplication in the image (MatMul).

2. **Dot Product of Q and K**: The query matrix Q is multiplied by the transpose of the key matrix K to obtain a score matrix that represents the relationship between different words (Step 2 in the image). This score matrix indicates how much focus should be put on other parts of the input sentence when encoding a particular word.

3. **Scaling**: The scores are scaled down by the square root of the dimension of the key vectors $ \sqrt{d_k} $ (Step 3 in the image). This is done to prevent the softmax from having an extremely small gradient during training, which could happen if the dot products are very large.

4. **Softmax**: A softmax function is applied to the scaled scores to obtain the weights on the values (Step 4 in the image). The softmax function turns the scores into probabilities that sum to one. Before the softmax, the scores of padded keys are set to $-\infty$ (see *Padding mask* below), so they receive zero weight.

5. **Output**: The softmax probabilities are then multiplied by the value matrix V (Step 5 in the image), and then the results of this multiplication are summed up to produce the final output for each word.

### Multi-Head Attention

The `MultiHeadAttention` class extends the attention mechanism to operate on multiple representations of the input simultaneously, allowing the model to capture a richer representation of the input data. This is achieved by splitting the input into multiple "heads," applying the dot product attention independently to each head, and then concatenating the results. This layer improves the model's ability to focus on different parts of the input sequence for different tasks.

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function, it linearly projects the queries, keys, and values multiple times with different, learned linear projections. This process involves:

1. **Linear Projections**: The queries, keys, and values are linearly projected $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions, respectively.

2. **Scaled Dot-Product Attention**: Each projected version of Q, K, and V computes the attention function in parallel, yielding $d_v$-dimensional output values: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

3. **Concatenation**: The $h$ parallel attention outputs are concatenated into a single matrix: $\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$, where:
   - $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$
   - $\text{Concat}(\text{head}_1, ..., \text{head}_h)$ represents the concatenation of the outputs from all the attention heads.

4. **Final Linear Transformation**: The concatenated matrix is again linearly transformed into the desired dimension.
   - $W^O$ is the final linear transformation matrix that projects the concatenated matrix back to the original embedding dimension.

The multi-head attention mechanism allows the model to capture information from different positions and representation subspaces, improving its ability to learn complex dependencies in the data.

Where $W_i^Q$, $W_i^K$, $W_i^V$, and $W^O$ are parameter matrices and $h$ is the number of heads.

Note that this code (like most papers and PyTorch) stores the tokens in the **rows** of the matrices, so it computes $\text{softmax}(QK^T/\sqrt{d_k})V$. The book stores tokens in columns and writes the same computation as $\mathbf{V}\cdot\text{Softmax}[\mathbf{K}^T\mathbf{Q}/\sqrt{D_q}]$: every matrix is transposed and the products appear in the reverse order.

### Padding mask
The reviews in a batch have different lengths, so the shorter ones are padded with `<pad>` tokens. Without further care, every token would also attend to these meaningless padding tokens, and the result for a review would depend on how long the other reviews in its batch are. We therefore build a boolean `key_padding_mask` of shape (batch, sequence length) that is `True` at padded positions. Inside `DotProductAttention` the scores for padded keys are replaced by $-\infty$ before the softmax, so every token attends only to the real tokens of its own review.

### TransformerBlock
The `TransformerBlock` encapsulates a single transformer layer, which consists of a multi-head attention layer (our own `MultiHeadAttention` class above) followed by a position-wise feedforward network. Each of these components is followed by a residual connection and layer normalization. This block is the fundamental building block of the transformer model, enabling it to capture complex dependencies in the data.

An earlier version of this notebook used PyTorch's `nn.MultiheadAttention(embed_dim, num_heads)` here. By default that layer expects tensors of shape (sequence length, batch, embedding), but it was given tensors of shape (batch, sequence length, embedding). It therefore treated the batch dimension as the sequence: each token attended to the tokens *at the same position in the other reviews of the batch*, not to the other tokens of its own review. The model still learned something (the embeddings and the pooling do most of the work for this task), but it was not a transformer over the words of a review. Passing `batch_first=True` would also fix this; here we use the notebook's own attention classes so that you can see every step.

### TokenAndPositionEmbedding
The `TokenAndPositionEmbedding` class is responsible for converting input tokens into embeddings and adding position information. This is crucial for the model to understand the order of tokens in the input sequence, as the transformer architecture itself does not have any inherent notion of sequence order: self-attention is equivariant to permutations of its inputs (notebook 12.1). The flag `use_pos` lets us switch the position embedding off, which we use in the ablation at the end of the notebook.

### TransformerModel
Finally, the `TransformerModel` class combines all the components into a complete model. It starts with the token and position embeddings, followed by a sequence of transformer blocks. The output of the last transformer block is then passed through a dropout layer, averaged over the real (non-padded) tokens of each review (*masked mean pooling*), and passed through a linear layer to produce the final class predictions (two logits, one for each class). This model is capable of capturing complex patterns in text data, making it well-suited for sentiment classification tasks.

This implementation showcases how to build a transformer model from scratch using PyTorch, including custom attention mechanisms and embedding layers, for sentiment classification on text data.

In [4]:
class DotProductAttention(nn.Module):
    def __init__(self, use_scale):
        super(DotProductAttention, self).__init__()
        self.use_scale = use_scale

    def forward(self, query, key, value, key_padding_mask=None):
        d_k = query.size(-1)  # Automatically determine the dimension of the queries and keys from the input shape
        scale = 1. / np.sqrt(d_k) if self.use_scale else 1.

        scores = torch.matmul(query, key.transpose(-2, -1)) * scale  # (batch, n_queries, n_keys)
        if key_padding_mask is not None:
            # key_padding_mask is (batch, n_keys) and True at padded positions: give them zero attention
            scores = scores.masked_fill(key_padding_mask[:, None, :], float('-inf'))
        attention = torch.softmax(scores, dim=-1)
        return torch.matmul(attention, value)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super(MultiHeadAttention, self).__init__()
        self.h = h
        self.d_model = d_model
        units = d_model // h

        self.layersQ = nn.ModuleList([nn.Linear(d_model, units, bias=False) for _ in range(h)])
        self.layersK = nn.ModuleList([nn.Linear(d_model, units, bias=False) for _ in range(h)])
        self.layersV = nn.ModuleList([nn.Linear(d_model, units, bias=False) for _ in range(h)])

        self.attention = DotProductAttention(use_scale=True)
        self.out = nn.Linear(units * h, d_model, bias=False)

    def forward(self, query, key, value, key_padding_mask=None):
        # Process each head separately
        qs, ks, vs = [], [], []
        for i in range(self.h):
            qs.append(self.layersQ[i](query))
            ks.append(self.layersK[i](key))
            vs.append(self.layersV[i](value))

        # Compute the attention output of each head
        outs = []
        for i in range(self.h):
            out = self.attention(qs[i], ks[i], vs[i], key_padding_mask)
            outs.append(out)

        # Concatenate all heads together
        concatenated = torch.cat(outs, dim=-1)

        # Final linear layer
        out = self.out(concatenated)

        return out


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(embed_dim, num_heads)  # Inputs are (batch, sequence, embedding)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.layernorm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)

    def forward(self, inputs, key_padding_mask=None):
        attn_output = self.att(inputs, inputs, inputs, key_padding_mask)  # Self-attention: query = key = value
        out1 = self.layernorm1(inputs + self.dropout1(attn_output))

        ffn_output = self.ffn(out1)
        out = self.layernorm2(out1 + self.dropout2(ffn_output))
        return out


class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, maxlen, vocab_size, embed_dim, use_pos=True):
        super(TokenAndPositionEmbedding, self).__init__()
        self.use_pos = use_pos
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)  # Token embedding. Learnable weights
        if use_pos:
            self.pos_emb = nn.Embedding(maxlen, embed_dim)  # Positional embedding. Learnable weights

    def forward(self, x):
        x_emb = self.token_emb(x)
        if self.use_pos:
            seq_len = x.shape[1]
            positions = torch.arange(seq_len, device=x.device)  # Generate positions on the same device as input
            positions = self.pos_emb(positions)[None, :]        # Embed and add a batch dimension
            x_emb = x_emb + positions
        return x_emb


class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, dropout_rate, maxlen, use_pos=True):
        super(TransformerModel, self).__init__()
        self.embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim, use_pos)
        self.transformer_blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        key_padding_mask = (x == PAD_IDX)  # True at padded positions, shape (batch, sequence)
        x = self.embedding(x)
        for transformer in self.transformer_blocks:
            x = transformer(x, key_padding_mask)
        x = self.dropout(x)
        # Masked mean pooling: average only over the real tokens of each review
        keep = (~key_padding_mask).unsqueeze(-1).float()  # (batch, sequence, 1)
        x = (x * keep).sum(dim=1) / keep.sum(dim=1)
        return self.out(x)

Before training, let's check that the attention really works across the tokens of each review and that the padding is ignored. We run one review on its own, and then the same review in a batch together with a much longer review (so that it gets padded). With a correct padding mask, the logits for the short review must be identical in both cases. With the old `nn.MultiheadAttention` bug, the output of a review depended on the other reviews in the batch.

In [5]:
torch.manual_seed(0)
check_model = TransformerModel(vocab_size, 16, 2, 32, 1, 2, 0.0, MAXLEN).to(device).eval()
short_review = encode("A wonderful, moving film.")
long_review = encode("This was one of the longest and most boring movies I have ever seen, and I would not recommend it to anyone at all.")
with torch.no_grad():
    alone = check_model(short_review.unsqueeze(0).to(device))
    batch = pad_sequence([short_review, long_review], batch_first=True, padding_value=PAD_IDX).to(device)
    in_batch = check_model(batch)[0:1]
print("Short review alone:        ", alone.cpu().numpy())
print("Short review in a batch:   ", in_batch.cpu().numpy())
print("Identical (padding ignored):", torch.allclose(alone, in_batch, atol=1e-5))

Short review alone:         [[-0.12273283 -0.19913167]]
Short review in a batch:    [[-0.12273283 -0.19913161]]
Identical (padding ignored): True


## Step 4: Instantiate the Model, Loss, and Optimizer

In this section, we initialize the transformer model with specific hyperparameters and prepare it for training. This involves defining the model architecture, setting up the optimizer, and specifying the loss function.

### Model Configuration
- `MAXLEN` (defined in Step 2) sets the maximum sequence length of a review, and therefore the number of learned position embeddings.
- `VOCAB_SIZE` is the total number of unique tokens in our vocabulary.
- `EMBED_DIM` specifies the dimensionality of the word embeddings.
- `NUM_HEADS` determines the number of attention heads in the multi-head attention layer (each head has dimension `EMBED_DIM // NUM_HEADS`).
- `FF_DIM` is the size of the hidden layer in the feedforward network within each transformer block.
- `NUM_LAYERS` indicates how many transformer blocks to stack in the model.
- `NUM_CLASSES` is set to 2, corresponding to positive and negative sentiment classes.
- `DROPOUT_RATE` controls the dropout rate to prevent overfitting.

### Model Initialization
The `TransformerModel` is initialized with the specified hyperparameters to create an instance of our sentiment classification model. We also count its trainable parameters. As for the LSTM, most of them are in the token embedding table.

### Optimizer and Loss Function
- The Adam optimizer (learning rate $10^{-3}$) is chosen for its efficiency and effectiveness in handling sparse gradients and adapting the learning rate for each parameter.
- The model outputs two logits (one per class), so we use the multiclass cross-entropy loss `nn.CrossEntropyLoss` (chapter 5 of the book). With two classes this is equivalent to the binary cross-entropy used in notebook 12.5.

This setup is crucial for training the transformer model on the sentiment classification task, enabling it to learn from the training data and make accurate predictions.

In [6]:
VOCAB_SIZE = vocab_size
EMBED_DIM = 64              # Size of each word embedding
NUM_HEADS = 2               # Number of attention heads
FF_DIM = 128                # Feedforward network hidden units
NUM_LAYERS = 1              # Number of transformer layers
NUM_CLASSES = 2             # Positive or negative
DROPOUT_RATE = 0.2
LEARNING_RATE = 1e-3


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


torch.manual_seed(SEED)
model = TransformerModel(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS, NUM_CLASSES, DROPOUT_RATE, MAXLEN).to(device)
print(model)
n_params = count_parameters(model)
print(f"Trainable parameters: {n_params:,}")
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

TransformerModel(
  (embedding): TokenAndPositionEmbedding(
    (token_emb): Embedding(20002, 64, padding_idx=0)
    (pos_emb): Embedding(500, 64)
  )
  (transformer_blocks): ModuleList(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (layersQ): ModuleList(
          (0-1): 2 x Linear(in_features=64, out_features=32, bias=False)
        )
        (layersK): ModuleList(
          (0-1): 2 x Linear(in_features=64, out_features=32, bias=False)
        )
        (layersV): ModuleList(
          (0-1): 2 x Linear(in_features=64, out_features=32, bias=False)
        )
        (attention): DotProductAttention()
        (out): Linear(in_features=64, out_features=64, bias=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=64, bias=True)
      )
      (layernorm1): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
      (layernorm2): LayerNorm((64,), 

## Step 5: Training and Validation Loops

This section outlines the core of our model training and evaluation process. It includes two distinct parts: the training loop and the validation loop.

- **Training Loop**: The training loop is where the model learns from the training dataset. It involves setting the model to training mode, iterating over the training data, computing the loss (using the specified criterion), and updating the model parameters based on the computed gradients. As in notebook 12.5, we clip the norm of the gradient to 1.0. This loop also calculates and accumulates the training loss and accuracy to monitor the performance of the model during training.

- **Validation Loop**: The validation loop is executed after the training loop and is used to evaluate the model's performance on the held-out validation set, which it does not see during training. The model is set to evaluation mode, which disables certain layers/functions like dropout, specific to training. Similar to the training loop, it computes the loss and accuracy but does not update the model's parameters, and it runs inside `torch.no_grad()`. This step is crucial for assessing the model's generalization ability and preventing overfitting.

- **Epoch Iteration**: Both loops are encapsulated within an epoch iteration loop (the function `fit`), where an epoch represents a single pass through the entire training dataset. After each epoch, the model's performance is assessed using both the training and validation data, and we keep a copy of the weights from the epoch with the best validation accuracy. We wrap the epoch loop in a function because we will train a second model in the ablation below.

In [7]:
# Training loop
def train(model, dataloader, optimizer, criterion, device):
    model.train()  # Set the model to training mode
    epoch_loss = 0
    correct = 0
    total = 0
    for labels, texts, lengths in dataloader:
        labels = labels.to(device)
        texts = texts.to(device)

        optimizer.zero_grad()  # Clear the gradients
        predictions = model(texts)  # Forward pass: compute the output
        loss = criterion(predictions, labels)  # Compute the loss
        loss.backward()  # Backward pass: compute the gradient of the loss with respect to model parameters
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clip the gradient norm
        optimizer.step()  # Update the model parameters

        epoch_loss += loss.item()  # Accumulate the loss
        correct += (predictions.argmax(1) == labels).type(torch.float).sum().item() # Count the number of correct predictions
        total += labels.size(0)

    accuracy = correct / total  # Compute the accuracy
    return epoch_loss / len(dataloader), accuracy  # Return the average loss and accuracy


# Validation loop
def evaluate(model, dataloader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    epoch_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():  # No gradients needed
        for labels, texts, lengths in dataloader:
            labels = labels.to(device)
            texts = texts.to(device)

            predictions = model(texts)  # Forward pass
            loss = criterion(predictions, labels)  # Compute loss

            epoch_loss += loss.item()  # Accumulate the loss
            correct += (predictions.argmax(1) == labels).type(torch.float).sum().item()
            total += labels.size(0)

    accuracy = correct / total  # Compute the accuracy
    return epoch_loss / len(dataloader), accuracy  # Return the average loss and accuracy


# Epoch loop: train, validate, and keep the weights of the best validation epoch
def fit(model, optimizer, criterion, n_epochs):
    best_valid_accuracy, best_epoch, best_state = 0.0, 0, None
    start_time = time.time()
    for epoch in range(n_epochs):
        train_loss, train_accuracy = train(model, train_dataloader, optimizer, criterion, device)
        valid_loss, valid_accuracy = evaluate(model, valid_dataloader, criterion, device)
        if valid_accuracy > best_valid_accuracy:
            best_valid_accuracy, best_epoch = valid_accuracy, epoch + 1
            best_state = copy.deepcopy(model.state_dict())  # Keep the best weights in memory
        print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.3f}, Train Acc: {train_accuracy*100:.2f}%, Validation Loss: {valid_loss:.3f}, Validation Acc: {valid_accuracy*100:.2f}%')
    train_time = time.time() - start_time
    model.load_state_dict(best_state)  # Restore the best weights
    print(f"Training time: {train_time:.1f} s")
    print(f"Best validation accuracy: {best_valid_accuracy*100:.2f}% (epoch {best_epoch})")
    return best_valid_accuracy, best_epoch, train_time


N_EPOCHS = 5
best_valid_accuracy, best_epoch, train_time = fit(model, optimizer, criterion, N_EPOCHS)

Epoch: 1, Train Loss: 0.536, Train Acc: 71.67%, Validation Loss: 0.412, Validation Acc: 81.00%


Epoch: 2, Train Loss: 0.336, Train Acc: 85.61%, Validation Loss: 0.344, Validation Acc: 84.96%


Epoch: 3, Train Loss: 0.250, Train Acc: 90.04%, Validation Loss: 0.335, Validation Acc: 86.12%


Epoch: 4, Train Loss: 0.190, Train Acc: 92.68%, Validation Loss: 0.353, Validation Acc: 86.48%


Epoch: 5, Train Loss: 0.136, Train Acc: 95.15%, Validation Loss: 0.416, Validation Acc: 86.40%
Training time: 81.7 s
Best validation accuracy: 86.48% (epoch 4)


## Step 6: Evaluate on the Test Set

The `fit` function has restored the weights from the epoch with the best validation accuracy. We now evaluate the model on the test set, once.

In [8]:
test_loss, test_accuracy = evaluate(model, test_dataloader, criterion, device)
print(f'Test Loss: {test_loss:.3f}, Test Acc: {test_accuracy*100:.2f}%')
print(f"Trainable parameters: {n_params:,}, epochs: {N_EPOCHS}, training time: {train_time:.1f} s on {device}")

Test Loss: 0.388, Test Acc: 84.76%
Trainable parameters: 1,345,474, epochs: 5, training time: 81.7 s on cuda


## Step 7: What if we remove the position embeddings?

Without position embeddings, the transformer has no way of knowing the order of the tokens: self-attention is equivariant to permutations of its inputs, the feedforward network acts on each token separately, and the mean pooling ignores the order too. The model then sees each review as a *bag of words* (a multiset of tokens), and shuffling the words of a review would not change its prediction at all.

TODO: Before running the next cell, predict how much the test accuracy will drop when we train the same model without the position embeddings (`use_pos=False`). A lot, a little, or not at all? Why?

The cell below trains a model that is identical except for the flag `use_pos=False`, with the same seed, number of epochs, and optimizer settings.

In [9]:
torch.manual_seed(SEED)
model_nopos = TransformerModel(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS, NUM_CLASSES, DROPOUT_RATE, MAXLEN, use_pos=False).to(device)
n_params_nopos = count_parameters(model_nopos)
print(f"Trainable parameters: {n_params_nopos:,} (the {MAXLEN} x {EMBED_DIM} position embedding is gone)")
optimizer_nopos = optim.Adam(model_nopos.parameters(), lr=LEARNING_RATE)
best_valid_accuracy_nopos, best_epoch_nopos, train_time_nopos = fit(model_nopos, optimizer_nopos, criterion, N_EPOCHS)

test_loss_nopos, test_accuracy_nopos = evaluate(model_nopos, test_dataloader, criterion, device)
print(f'Test Loss: {test_loss_nopos:.3f}, Test Acc: {test_accuracy_nopos*100:.2f}%')

Trainable parameters: 1,313,474 (the 500 x 64 position embedding is gone)


Epoch: 1, Train Loss: 0.526, Train Acc: 72.90%, Validation Loss: 0.440, Validation Acc: 80.16%


Epoch: 2, Train Loss: 0.325, Train Acc: 86.32%, Validation Loss: 0.330, Validation Acc: 85.76%


Epoch: 3, Train Loss: 0.251, Train Acc: 89.87%, Validation Loss: 0.320, Validation Acc: 87.60%


Epoch: 4, Train Loss: 0.199, Train Acc: 92.36%, Validation Loss: 0.340, Validation Acc: 87.56%


Epoch: 5, Train Loss: 0.154, Train Acc: 94.26%, Validation Loss: 0.348, Validation Acc: 87.40%
Training time: 81.4 s
Best validation accuracy: 87.60% (epoch 3)


Test Loss: 0.346, Test Acc: 85.81%


In [10]:
# The model without position embeddings really ignores word order: shuffling the words does not change its output
review = "The acting was not good, and the plot was not interesting at all."
ids = encode(review)
shuffled = ids[torch.randperm(len(ids))]
print("Original:", " ".join(itos[i] for i in ids))
print("Shuffled:", " ".join(itos[i] for i in shuffled))
for name, m in [("with positions", model), ("without positions", model_nopos)]:
    m.eval()
    with torch.no_grad():
        p_orig = torch.softmax(m(ids.unsqueeze(0).to(device)), dim=-1)[0, 1].item()
        p_shuf = torch.softmax(m(shuffled.unsqueeze(0).to(device)), dim=-1)[0, 1].item()
    print(f"Model {name:18s}: P(positive) original = {p_orig:.4f}, shuffled = {p_shuf:.4f}")

Original: the acting was not good , and the plot was not interesting at all .
Shuffled: was was at not good the interesting . the not all and , acting plot
Model with positions    : P(positive) original = 0.0000, shuffled = 0.0003
Model without positions : P(positive) original = 0.0033, shuffled = 0.0033


### Discussion: why does word order matter so little here?

In our run, the model without position embeddings reached a test accuracy of 85.81%, compared with 84.76% for the model with position embeddings. Removing all information about word order did not hurt at all; the small difference in the other direction comes from a single training run of each model and should not be over-interpreted. The last cell confirms that the model without positions really ignores the order: it gives exactly the same probability to the original and the shuffled review.

This is a property of the task more than of the model. The sentiment of a movie review is mostly carried by *which* words appear ("great", "wonderful", "boring", "worst", "waste"), not by the order in which they appear, so bag-of-words cues dominate. Classical classifiers that only count words (for example logistic regression on word counts) already reach the high 80s on IMDB. A review is also long (the median is about 200 tokens here), so the occasional phrase whose meaning depends on order, such as "not good" or "I expected to hate it, but I loved it", is usually outweighed by the many other sentiment words in the review. With only 22,500 training reviews and 5 epochs, the model has little opportunity to learn to exploit order anyway.

Word order is essential for many other tasks: in language modeling, translation, or question answering, "the dog bit the man" and "the man bit the dog" must be treated differently. There, removing the position encodings is disastrous. So a good result on one benchmark does not mean that a component is useless in general, and ablations like this one tell you as much about the task as about the model.

TODO: Try to find reviews where the order matters (for example "not bad, actually good" versus "not good, actually bad") and compare the predictions of the two models.

## Step 8: Comparison with the LSTM

The cell below collects the results. The LSTM numbers are copied from the executed notebook 12.5, which uses the same data, vocabulary, split, and number of epochs (rerun notebook 12.5 and update them if you change anything).

In [11]:
# Copied from the output of notebook 12.5
lstm_params, lstm_epochs, lstm_test_accuracy = 2_692_481, 5, 0.8472

print(f"{'Model':38s} {'Trainable parameters':>21s} {'Epochs':>7s} {'Test accuracy':>14s}")
print(f"{'LSTM (notebook 12.5)':38s} {lstm_params:>21,} {lstm_epochs:>7d} {lstm_test_accuracy*100:>13.2f}%")
print(f"{'Transformer':38s} {n_params:>21,} {N_EPOCHS:>7d} {test_accuracy*100:>13.2f}%")
print(f"{'Transformer without position embedding':38s} {n_params_nopos:>21,} {N_EPOCHS:>7d} {test_accuracy_nopos*100:>13.2f}%")

Model                                   Trainable parameters  Epochs  Test accuracy
LSTM (notebook 12.5)                               2,692,481       5         84.72%
Transformer                                        1,345,474       5         84.76%
Transformer without position embedding             1,313,474       5         85.81%


### Results

| Model | Trainable parameters | Epochs | Test accuracy |
|---|---:|---:|---:|
| LSTM (notebook 12.5), embedding 128, hidden 128 | 2,692,481 | 5 | 84.72% |
| Transformer, embedding 64, 2 heads, 1 layer | 1,345,474 | 5 | 84.76% |
| Transformer without position embedding | 1,313,474 | 5 | 85.81% |

(Numbers from the runs in notebooks 12.5 and 12.6 on one GPU; your numbers will vary slightly with the hardware and library versions. On our GPU, training took 72 s for the LSTM and 82 s for each transformer.)

All three models reach roughly the same test accuracy, about 85%. In all of them, most of the parameters are in the token embedding table (20,002 x 128 = 2,560,256 for the LSTM and 20,002 x 64 = 1,280,128 for the transformers), so the sequence models themselves are tiny. On this small dataset and short training budget, the transformer's ability to relate any two tokens directly does not give it an advantage over the LSTM, and, as the ablation shows, the task can be solved well without any notion of word order. The advantages of transformers (parallel processing of the whole sequence, direct long-range interactions, and above all the ability to scale to huge datasets and models) show up in pretraining on large corpora. In notebook 12.8, a pretrained DistilBERT model fine-tuned on only 2,000 of these reviews (less than a tenth of our training set) reaches about 88% test accuracy, and even a logistic regression on its frozen features matches our models trained from scratch.